# Distillation: η αεροπορική Nimbra Air

**Τι είναι το distillation;**
Ένα μεγάλο, ακριβό μοντέλο (ο **teacher**) «μεταφέρει» γνώση σε ένα μικρό, φθηνό μοντέλο (τον **student**).
Ο teacher γράφει παραδείγματα (ερωτήσεις + απαντήσεις) και ο student εκπαιδεύεται πάνω τους (fine-tuning),
ώστε να απαντάει όπως ο teacher, χωρίς να χρειαζόμαστε πια τον teacher.

**Το σενάριο**
Η **Nimbra Air** είναι μια **φανταστική** αεροπορική εταιρεία.
Ο **teacher** (μεγάλο μοντέλο στο Azure OpenAI) έχει το εγχειρίδιο της εταιρείας: αποσκευές, ναύλους, check-in, κατοικίδια, πόντους.
Ο **student** (Qwen2.5 0.5B) δεν μπορεί να ξέρει τίποτα από αυτά, γιατί δεν υπάρχουν πουθενά στο internet.

Ο student **δεν βλέπει ποτέ** το εγχειρίδιο. Μαθαίνει μόνο από τις ερωταπαντήσεις του teacher.

**Τα βήματα**
1. Διαβάζουμε το εγχειρίδιο (`airline_facts.txt`).
2. Ο teacher γράφει ερωταπαντήσεις και τις αποθηκεύει **όλες σε ένα αρχείο** (`airline_data.jsonl`).
3. Ο student χωρίζει το αρχείο σε **train** (90%) και **test** (10%) και βαθμολογείται **πριν** την εκπαίδευση.
4. Εκπαιδεύουμε τον student και τον βαθμολογούμε ξανά **μετά**.
5. Ρωτάμε τον αποθηκευμένο student ό,τι θέλουμε.

## 0. Εγκατάσταση βιβλιοθηκών

- `langchain`, `langchain-openai`: για να μιλάμε με τον teacher (Azure OpenAI).
- `datasets`: για να φορτώσουμε τα δεδομένα και να τα χωρίσουμε σε train / test.
- `trl`, `transformers`: για το fine-tuning του student.

Αν μετά την εγκατάσταση σου ζητηθεί **Restart session**, κάνε restart και συνέχισε από το επόμενο κελί.

In [ ]:
pip install -U langchain==1.5.0 langchain-core==1.8.1 langchain-openai==1.8.0 datasets trl transformers

## Ρυθμίσεις Colab

Πριν τρέξεις το επόμενο κελί:
1. **Secrets** (το εικονίδιο 🔑 αριστερά): πρόσθεσε τα `AZURE_OPENAI_DEPLOYMENT_NAME`, `AZURE_OPENAI_ENDPOINT`,
   `AZURE_OPENAI_API_KEY`, `OPENAI_API_VERSION` και ενεργοποίησε το «Notebook access».
2. **Runtime → Change runtime type → T4 GPU**.

Το κελί συνδέει το Google Drive, ώστε τα αρχεία (δεδομένα και εκπαιδευμένος student) να μη χαθούν όταν κλείσει το Colab.

In [ ]:
import os, torch
from google.colab import drive

# Συνδέουμε το Google Drive και δουλεύουμε μέσα στον φάκελο PwC
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/Colab Notebooks/PwC", exist_ok=True)
os.chdir("/content/drive/MyDrive/Colab Notebooks/PwC")

# Ελέγχουμε ότι υπάρχει GPU (χωρίς GPU η εκπαίδευση θα ήταν πολύ αργή)
assert torch.cuda.is_available(), "Δεν βρέθηκε GPU: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 1. Το εγχειρίδιο της εταιρείας (η «γνώση» του teacher)

Τα γεγονότα βρίσκονται στο αρχείο `airline_facts.txt`, ένα σε κάθε γραμμή.
Είναι στα αγγλικά, γιατί το μικρό μοντέλο τα πάει πολύ καλύτερα στα αγγλικά.
Μπορείς να αλλάξεις ή να προσθέσεις γεγονότα στο αρχείο χωρίς να αγγίξεις τον κώδικα.

In [ ]:
# Το airline_facts.txt πρέπει να είναι στον φάκελο PwC του Drive.
# Αν δεν είναι, ανέβασέ το από τον υπολογιστή σου (αποθηκεύεται στον τρέχοντα φάκελο, δηλαδή στο Drive):
# from google.colab import files
# files.upload()

# Μία γραμμή = ένα γεγονός. Οι κενές γραμμές και όσες ξεκινάνε με # αγνοούνται.
with open("airline_facts.txt", encoding="utf-8") as f:
    FACTS = [line.strip() for line in f if line.strip() and not line.startswith("#")]

# Όλο το εγχειρίδιο σε ένα κείμενο: αυτό θα το δίνουμε στον teacher
HANDBOOK = "\n".join(FACTS)
print(len(FACTS), "γεγονότα")
print(HANDBOOK[:300], "...")

## 2. Ο teacher φτιάχνει ερωταπαντήσεις (στα αγγλικά)

Για **κάθε** γεγονός ο teacher γράφει 8 διαφορετικές ερωτήσεις πελατών με τις απαντήσεις τους.
Έτσι ο student βλέπει το ίδιο γεγονός με πολλές διατυπώσεις και μαθαίνει **το γεγονός**, όχι μια συγκεκριμένη πρόταση.

Ζητάμε από τον teacher **δομημένη έξοδο** (structured output): αντί για ελεύθερο κείμενο,
επιστρέφει μια λίστα από αντικείμενα `QA` (ερώτηση + απάντηση), που τα διαβάζουμε εύκολα από τον κώδικα.

Όλες οι ερωταπαντήσεις γράφονται σε **ένα** αρχείο, το `airline_data.jsonl`.
Κάθε γραμμή έχει τη μορφή `{"prompt": "Question: ...\nAnswer:", "completion": " ..."}`, που είναι η μορφή που περιμένει ο `SFTTrainer`.
Ο χωρισμός σε train και test γίνεται αργότερα, στο βήμα του student.

In [ ]:
import json
from pydantic import BaseModel
from langchain_openai import AzureChatOpenAI
from google.colab import userdata

# Ο teacher: το μεγάλο μοντέλο στο Azure OpenAI (τα στοιχεία σύνδεσης έρχονται από τα Secrets)
teacher = AzureChatOpenAI(
    azure_deployment=userdata.get("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=userdata.get("AZURE_OPENAI_ENDPOINT"),
    api_key=userdata.get("AZURE_OPENAI_API_KEY"),
    api_version=userdata.get("OPENAI_API_VERSION"),
)

# Η μορφή της απάντησης που θέλουμε από τον teacher (structured output)
class QA(BaseModel):
    question: str
    answer: str

class QAList(BaseModel):
    items: list[QA]

# Ένα prompt για κάθε γεγονός του εγχειριδίου
prompts = [
    f"""You work in customer service for the airline Nimbra Air. This is the company handbook:
{HANDBOOK}

Write 8 different questions, in English, that a customer might ask, whose answer is this fact:
"{fact}"
Every question must mention "Nimbra Air" (or "Nimbra Miles" / "NimbraGo").
Vary the wording a lot (how much / can I / what / when / who / is it true that..., short and long questions).
Answer each question in English, in ONE short sentence that clearly states the fact. Use only the handbook."""
    for fact in FACTS
]

# Στέλνουμε όλα τα prompts μαζί (έως 5 ταυτόχρονα)
results = teacher.with_structured_output(QAList).batch(prompts, config={"max_concurrency": 5})

# Ο teacher γράφει ΟΛΕΣ τις ερωταπαντήσεις σε ένα αρχείο
with open("airline_data.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        for qa in r.items:
            row = {"prompt": f"Question: {qa.question}\nAnswer:", "completion": " " + qa.answer.strip()}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

### Έλεγχος των δεδομένων

Πριν εκπαιδεύσουμε, κοιτάμε τι έγραψε ο teacher: πόσα παραδείγματα, και πώς διατύπωσε τις ερωτήσεις για ένα γεγονός.

In [ ]:
# Πόσες γραμμές έγραψε ο teacher και οι πρώτες 3
with open("airline_data.jsonl", encoding="utf-8") as f:
    lines = f.readlines()
print(len(lines), "παραδείγματα")
for line in lines[:3]:
    print(line.strip())

# Ίδιο γεγονός, διαφορετικές διατυπώσεις (άλλαξε το i για να δεις άλλο γεγονός)
i = 16
print("\nFACT:", FACTS[i], "\n")
for qa in results[i].items:
    print("Q:", qa.question)
    print("A:", qa.answer, "\n")

## 3. Ο student και ο χωρισμός σε train / test

Ο student φορτώνει το **ένα** αρχείο του teacher και το χωρίζει: **90% train** (για εκπαίδευση) και **10% test** (για έλεγχο).
Τα παραδείγματα του test ο student **δεν τα βλέπει** στην εκπαίδευση. Επειδή κάθε γεγονός έχει 8 διατυπώσεις,
στο test τον ρωτάμε συνήθως πράγματα που έμαθε, αλλά **με διαφορετικά λόγια**.

Βασικές ρυθμίσεις του `SFTConfig`:
- `num_train_epochs=10`: πόσες φορές περνάει από όλα τα δεδομένα. Νέα γνώση θέλει πολλές επαναλήψεις.
- `per_device_train_batch_size=8`: πόσα παραδείγματα ανά βήμα.
- `learning_rate=1e-4`: πόσο «μεγάλα» βήματα κάνει στην εκπαίδευση.
- `fp16=True`: μισή ακρίβεια για ταχύτητα (η T4 δεν υποστηρίζει bf16).
- `activation_checkpointing=True`: εξοικονομεί μνήμη GPU.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# Ο student φορτώνει το αρχείο του teacher και το χωρίζει σε train / test
dataset = load_dataset("json", data_files="airline_data.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=0)   # 90% train, 10% test

trainer = SFTTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",      # ο student: μικρό μοντέλο 0.5B παραμέτρων
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=SFTConfig(
        output_dir="student_airline",
        num_train_epochs=10,             # νέα γνώση θέλει περισσότερες επαναλήψεις από το «ύφος»
        per_device_train_batch_size=8,
        learning_rate=1e-4,
        fp16=True,                       # η T4 δεν υποστηρίζει bf16
        bf16=False,
        max_length=256,                  # μέγιστο μήκος παραδείγματος σε tokens
        logging_steps=10,                # τυπώνει το loss κάθε 10 βήματα
        save_strategy="no",              # δεν κρατάμε ενδιάμεσα checkpoints
        report_to="none",
        model_init_kwargs={"torch_dtype": torch.float32},
        activation_checkpointing=True,   # λιγότερη μνήμη GPU
    ),
)

## 4. Ο student ΠΡΙΝ την εκπαίδευση

Μετράμε δύο πράγματα:
- **Loss** στο test: πόσο «ξαφνιάζεται» το μοντέλο από τις απαντήσεις του teacher. Μικρότερο = καλύτερο.
- **Βαθμός από judge**: ένα μεγάλο μοντέλο (ο **judge**) συγκρίνει την απάντηση του student με την απάντηση του teacher και βάζει βαθμό 1–5.

Περιμένουμε ο student να «επινοεί» απαντήσεις, αφού δεν έχει ξανακούσει ποτέ για τη Nimbra Air.

In [ ]:
# Ο judge: το ίδιο μεγάλο μοντέλο, με temperature=0 για σταθερή βαθμολογία
judge = AzureChatOpenAI(
    azure_deployment=userdata.get("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=userdata.get("AZURE_OPENAI_ENDPOINT"),
    api_key=userdata.get("AZURE_OPENAI_API_KEY"),
    api_version=userdata.get("OPENAI_API_VERSION"),
    temperature=0.0,
)

def ask(prompt):
    """Ρωτάει τον student και επιστρέφει μόνο την απάντηση."""
    tok, model = trainer.processing_class, trainer.model
    model.eval()
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=60, do_sample=False)
    # Κρατάμε μόνο τα νέα tokens (όχι το prompt)
    ans = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    # Αν ο student συνεχίσει με νέα «Question:», την κόβουμε
    return ans.split("\nQuestion:")[0].strip()

def show(n=4):
    """Δείχνει δίπλα-δίπλα τις απαντήσεις student και teacher για τα πρώτα n του test."""
    for p, ref in zip(dataset["test"]["prompt"][:n], dataset["test"]["completion"][:n]):
        print(p)
        print("  student:", ask(p))
        print("  teacher:", ref.strip(), "\n")

def grade():
    """Ο judge βαθμολογεί (1–5) όλες τις απαντήσεις του student στο test και τυπώνει τον μέσο όρο."""
    scores = []
    for p, ref in zip(dataset["test"]["prompt"], dataset["test"]["completion"]):
        reply = judge.invoke(
            f"{p}\nReference answer: {ref.strip()}\nStudent answer: {ask(p)}\n"
            "The company is fictional, so judge ONLY by agreement with the reference answer. "
            "Rate the student answer from 1 to 5 (5 = same facts as the reference). "
            "Reply with one digit only."
        ).content.strip()
        if reply[:1].isdigit():
            scores.append(int(reply[0]))
    print("Μέσος βαθμός:", round(sum(scores) / len(scores), 2))

print("Loss πριν:", trainer.evaluate()["eval_loss"])
show()
grade()      # βαθμός ΠΡΙΝ την εκπαίδευση

## 5. Εκπαίδευση και ο student ΜΕΤΑ

Εκπαιδεύουμε τον student πάνω στο train, τον αποθηκεύουμε στο Drive (φάκελος `student_airline`)
και ξαναμετράμε loss και βαθμό στο **ίδιο** test. Αν η απόσταξη πέτυχε, το loss πέφτει και ο βαθμός ανεβαίνει.

In [ ]:
trainer.train()
trainer.save_model("student_airline")   # αποθήκευση στο Drive

print("Loss μετά:", trainer.evaluate()["eval_loss"])
show()
grade()      # βαθμός ΜΕΤΑ την εκπαίδευση

## 6. Ρώτα τον student ό,τι θέλεις

Ο student είναι αποθηκευμένος στο Drive (φάκελος `student_airline`).
Τον ξαναφορτώνουμε από τον δίσκο, ανεξάρτητα από τον trainer, για να δούμε ότι η αποθήκευση δουλεύει.

Η τελευταία ερώτηση δεν αφορά την εταιρεία: ελέγχουμε αν ο student **ξέχασε** όσα ήξερε πριν (catastrophic forgetting).

In [ ]:
from transformers import pipeline

# Φορτώνουμε τον εκπαιδευμένο student από τον δίσκο
pipe = pipeline("text-generation", model="student_airline", device=0)

def student(q):
    out = pipe(f"Question: {q}\nAnswer:", max_new_tokens=60, do_sample=False, return_full_text=False)
    return out[0]["generated_text"].split("\nQuestion:")[0].strip()

for q in ["Can I bring my cat on a Nimbra Air flight?",
          "How much does an extra suitcase cost at the airport with Nimbra Air?",
          "What is the capital of France?"]:            # ξέχασε ό,τι ήξερε πριν;
    print(q, "→", student(q))

# Προαιρετικά: κατέβασε τον student στον υπολογιστή σου
# !zip -qr student_airline.zip student_airline
# from google.colab import files
# files.download("student_airline.zip")